<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-02-grafos/14%20-%20Menor%20Caminho%20Dijkstra%20e%20Roteamento%20Dinamico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementação orientada a objetos com suporte a Min-Heap (heapq), recálculo dinâmico de pesos e extração da sequência completa de estações com a distância total acumulada.

In [1]:
import heapq
from typing import Dict, List, Tuple, Optional

class Dijkstra_AGV_Router:
    """
    Roteador Dinâmico para AGVs baseado no Algoritmo de Dijkstra.
    Suporta recálculo dinâmico de pesos em tempo de execução (Min-Heap).
    """
    def __init__(self):
        self.estacoes: set = set()
        self.adj: Dict[str, List[Tuple[str, float]]] = {}

    def adicionar_estacao(self, tag_estacao: str):
        """Cadastra um nó/waypoint no dígrafo de navegação."""
        if tag_estacao not in self.estacoes:
            self.estacoes.add(tag_estacao)
            self.adj[tag_estacao] = []

    def adicionar_corredor(self, origem: str, destino: str, custo_m: float):
        """Adiciona um arco ponderado (corredor) de origem para destino."""
        self.adicionar_estacao(origem)
        self.adicionar_estacao(destino)
        self.adj[origem].append((destino, custo_m))

    def atualizar_custo_corredor(self, origem: str, destino: str, novo_custo_m: float) -> bool:
        """
        Recálculo dinâmico de peso (ex: aumento de custo por congestionamento
        ou bloqueio temporário).
        """
        if origem in self.adj:
            for i, (vizinho, custo) in enumerate(self.adj[origem]):
                if vizinho == destino:
                    self.adj[origem][i] = (destino, novo_custo_m)
                    return True
        return False

    def calcular_menor_caminho(self, origem: str, destino: str) -> Tuple[Optional[List[str]], float]:
        """
        Executa o algoritmo de Dijkstra utilizando Min-Heap O((|V| + |E|) log |V|).
        Retorna (sequencia_de_estacoes, distancia_total_m).
        """
        if origem not in self.estacoes or destino not in self.estacoes:
            return None, float('inf')

        distancias = {node: float('inf') for node in self.estacoes}
        predecessores = {node: None for node in self.estacoes}

        distancias[origem] = 0.0
        min_heap = [(0.0, origem)]

        while min_heap:
            dist_atual, u = heapq.heappop(min_heap)

            if dist_atual > distancias[u]:
                continue

            if u == destino:
                break

            for vizinho, peso in self.adj.get(u, []):
                dist_alt = dist_atual + peso
                if dist_alt < distancias[vizinho]:
                    distancias[vizinho] = dist_alt
                    predecessores[vizinho] = u
                    heapq.heappush(min_heap, (dist_alt, vizinho))

        # Reconstrução do caminho
        caminho = []
        passo = destino
        if distancias[destino] == float('inf'):
            return None, float('inf')

        while passo is not None:
            caminho.append(passo)
            passo = predecessores[passo]
        caminho.reverse()

        return caminho, distancias[destino]


# Demonstração e Validação do Roteamento
if __name__ == "__main__":
    router = Dijkstra_AGV_Router()

    # 1. Cadastro dos Corredores e Custos Originais (Metros)
    corredores = [
        ("ST-01", "DOC-101", 10.0),
        ("ST-01", "ALM-201", 12.0),
        ("DOC-101", "ALM-201", 15.0),
        ("ALM-201", "AMO-301", 20.0),
        ("AMO-301", "R-101", 18.0),
        ("AMO-301", "DEP-401", 14.0),
        ("R-101", "DEP-401", 25.0),
        ("DEP-401", "ST-01", 30.0)
    ]

    for orig, dest, custo in corredores:
        router.adicionar_corredor(orig, dest, custo)

    print("=== TESTE 1: ROTA OTIMIZADA NOMINAL (DIJKSTRA) ===")
    caminho_otimo, dist_total = router.calcular_menor_caminho("ST-01", "DEP-401")
    print(f"Caminho Mínimo: {' -> '.join(caminho_otimo)}")
    print(f"Distância Total: {dist_total:.1f} metros\n")

    # 2. Simulação de Evento Dinâmico: Congestionamento no Corredor AMO-301 -> DEP-401
    print("=== TESTE 2: RECÁLCULO DINÂMICO APÓS EVENTO DE TRÁFEGO ===")
    print("[EVENTO SCADA] Congestionamento detectado no corredor AMO-301 -> DEP-401. Custo alterado de 14m para 50m.")
    router.atualizar_custo_corredor("AMO-301", "DEP-401", 50.0)

    caminho_recalculado, nova_dist = router.calcular_menor_caminho("ST-01", "DEP-401")
    print(f"Novo Caminho Mínimo: {' -> '.join(caminho_recalculado)}")
    print(f"Nova Distância Total: {nova_dist:.1f} metros")

=== TESTE 1: ROTA OTIMIZADA NOMINAL (DIJKSTRA) ===
Caminho Mínimo: ST-01 -> ALM-201 -> AMO-301 -> DEP-401
Distância Total: 46.0 metros

=== TESTE 2: RECÁLCULO DINÂMICO APÓS EVENTO DE TRÁFEGO ===
[EVENTO SCADA] Congestionamento detectado no corredor AMO-301 -> DEP-401. Custo alterado de 14m para 50m.
Novo Caminho Mínimo: ST-01 -> ALM-201 -> AMO-301 -> R-101 -> DEP-401
Nova Distância Total: 75.0 metros
